# Lab06 - Training a deep neural network


#### Objectives:
1. Learn how to build a deep regular neural network
2. Identify the factors that affects the training of a deep regular neural network
    1. Initialization strategy (zeros, normal, xavier)
        <br> `nn.init.zeros_`, `nn.init.normal_`, `nn.init.xavier_normal_`, `nn.init.kaiming_normal_`
    2. Initializing a deep neural network by iterating all modules
        <br> `self.modules`, `isinstance`
    2. Activation (Sigmoid vs ReLU)
        <br> `torch.relu`, `torch.sigmoid`
    3. Batch Normalization
        <br> `torch.nn.BatchNorm1d`
    4. Dropout
        <br> `torch.nn.Dropout`
    
#### Reference:
[PyTorch Official Site](https://pytorch.org/docs/stable/nn.init.html)

#### Content:
1. [Loading CIFAR10](#1.-Loading-CIFAR10)
2. [Training and evaluation functions](#2.-Training-and-evaluation-functions)
3. [Initialization strategies](#3.-Initialization-strategies)
    1. [Model 1: Shallow, Init=zeros](#Model-1:-Shallow,-Init-=-zeros)
    2. [Model 2:  Shallow, Init=normal](#Model-2:-Shallow,-Init-=-normal)
    3. [Model 3:  Shallow, Init = xavier_normal](#Model-3:-Shallow,-Init-=-xavier_normal)
    4. [Model 4: Shallow, Init = kaiming_normal](#Model-4:-Shallow,-Init-=-kaiming_normal)
    5. [Model 5: Deep, Init = xavier_normal](#Model-5:-Deep,-Init-=-kaiming_normal)
4. [Batch Normalization](#4.-Batch-Normalization)
    1. [Model 6: Deep, BatchNorm, Sigmoid Activation](#Model-6:-Deep,-BatchNorm,-Sigmoid-Activation)
5. [ReLU Activation](#5.-ReLU-Activation)
    1. [Model 7:  Deep, ReLU Activation](#Model-7:-Deep,-ReLU-Activation)
    6. [Model 8:  Deep, BatchNorm, ReLU Activation](#Model-8:-Deep,-BatchNorm,-ReLU-Activation)
6. [Dropout](#6.-Dropout)
    1. [Model 9:  Deep, Dropout, ReLU Activation](#Model-9:-Deep,-Dropout,-ReLU-Activation)


## Introduction

Adding more blocks to a neural network makes it more difficult to train. In this practical we shall perform a series of experiments to evaluate the factors that may impact the training of neural network as more blocks are added. We shall also learn several techniques that help us to alleviate the problem. Specifically, we are going to build the following models:


<center><b>Table 1</b>: Result sheet for different configurations</center>

| Model |Evaluation|  #Blocks | Activation | Initialization | BN? | Dropout? | #Epochs | Accuracy |
|:---|:---|:--:|:---:|:---:|:---:|:---:|:---:|:---:|
|1 |Shallow, Initialization|  2 | Sigmoid | <font color="blue"><b>zeros</b></font> | - | - | <font color ="red">**?**</font> | <font color ="red">**?**</font>|
|2|Shallow, Initialization|  2 | Sigmoid | <font color="blue"><b>normal</b></font> | - | - | <font color ="red">**?**</font> | <font color ="red">**?**</font>|
|3|Shallow, Initialization|  2 | Sigmoid | <font color="blue"><b>xavier normal</b></font>  | - | - | <font color ="red">**?**</font> | <font color ="red">**?**</font>|
|4|Shallow, Initialization|  2 | Sigmoid | <font color="blue"><b>kaiming normal</b></font> | - | - | <font color ="red">**?**</font> | <font color ="red">**?**</font>|
|5|<font color="blue"><b>Deep</b></font>, Initialization|  <font color="blue"><b>6</b></font> | Sigmoid | kaiming normal | -- | - | <font color ="red">**?**</font> | <font color ="red">**?**</font>|
|6|Deep, BN|  6 | Sigmoid | kaiming normal | <font color="blue"><b>Yes</b></font> | - | <font color ="red">**?**</font> | <font color ="red">**?**</font>|
|7|Deep, ReLU|  6 | <font color="blue"><b>ReLU</b></font> | kaiming normal | - | - | <font color ="red">**?**</font> | <font color ="red">**?**</font>|
|8|Deep, BN+ReLU|  6 | <font color="blue"><b>ReLU</b></font> | kaiming normal | <font color="blue"><b>Yes</b></font> |- | <font color ="red">**?**</font> | <font color ="red">**?**</font>|
|9|Deep, Dropout+ReLU| 6 | ReLU | kaiming normal | Yes | <font color="blue"><b>Yes</b></font> | <font color ="red">**?**</font> | <font color ="red">**?**</font>|

* Model 1 to 4 show the impact of different initialization methods on regular neural networks. Both **xavier normal** and **kaiming normal** initialization methods performs better compared to **zeros** or **normal** initialization
* Model 5 shows that even the kaiming normal (and xavier normal) starts to work poorly when the network grows **deeper** from 2 to 6 blocks. The network cannot train properly anymore when it gets deeper.
* Model 6 shows that adding batch normalization or **batchnorm** allows us to train deeper network
* Model 7 shows that changing the activation from sigmoid to **relu** also allows us to train deeper network.
* Model 8 combines **relu** + **batchnorm** to get a better performance than model 6 and 7.
* Model 9 uses **dropout** to perform regularization.

Mount google drive to colab

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
cd "gdrive/My Drive/UCCD3074_Lab4"

Import required libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms
import math

from cifar10 import CIFAR10

%load_ext autoreload
%autoreload 2

Create a seed function to ensure the result is reproducible

In [ ]:
def random_seed():
    np.random.seed(0)  
    torch.manual_seed(0)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

## 1. Loading CIFAR10

The following code loads the dataset

In [ ]:
# transform the model
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# dataset
trainset = CIFAR10(train=True,  transform=transform, num_samples=10000)
testset  = CIFAR10(train=False,  transform=transform, num_samples=2000)

# dataloader]
trainloader = DataLoader(trainset, batch_size=32, shuffle=True, num_workers=4)
testloader  = DataLoader(testset, batch_size=128, shuffle=True, num_workers=4)

## 2. Training and evaluation functions

The `train` function trains the network `net` for `max_epochs` epochs. The training will stop running when the training loss saturates.

In [ ]:
def train(net, max_epochs=100, use_gpu=True):
    
    loss_iterations = int(np.ceil(len(trainloader)/3))
    stagnant_count  = 0 
    best_loss       = math.inf

    # transfer model to GPU
    if use_gpu and torch.cuda.is_available():
        net = net.cuda()
    
    # set the optimizer
    optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.9)
    
    # set to training mode
    net.train()

    # train the network    
    for e in range(max_epochs):    

        running_loss    = 0
        running_count   = 0
        
        for i, (inputs, labels) in enumerate(trainloader):

            # Clear all the gradient to 0
            optimizer.zero_grad()

            # transfer data to GPU
            if use_gpu and torch.cuda.is_available():
                inputs = inputs.cuda()
                labels = labels.cuda()

            # forward propagation to get h
            outs = net(inputs)

            # compute loss 
            loss = F.cross_entropy(outs, labels)

            # backpropagation to get dw
            loss.backward()

            # update w
            optimizer.step()

            # get the loss
            running_loss += loss.item()
            running_count += 1

             # display the averaged loss value 
            if i % loss_iterations == loss_iterations-1 or i == len(trainloader) - 1:                
                train_loss = running_loss / running_count
                running_loss = 0. 
                running_count = 0.
                print(f'[Epoch {e+1:2d} Iter {i+1:5d}/{len(trainloader)}]: train_loss = {train_loss:.4f}')       
                
                if train_loss >= best_loss:
                    stagnant_count += 1
                else:
                    best_loss = train_loss
                    stagnant_count = 0
                    
                if stagnant_count >= 3:
                    return


The `evaluate` function evaluates a trained model

In [ ]:
def evaluate(net, use_gpu=True):
    
    # set to evaluation mode
    net.eval() 

    # running_correct
    running_corrects = 0

    for inputs, targets in testloader:

        # transfer to the GPU
        if use_gpu and torch.cuda.is_available():
            inputs = inputs.cuda()  
            targets = targets.cuda()

        # perform prediction (no need to compute gradient)
        with torch.no_grad():
            outputs = net(inputs)  
            _, predicted = torch.max(outputs, 1)

            running_corrects += (targets == predicted).double().sum()


    print('Accuracy = {:.2f}%'.format(100*running_corrects/len(testloader.dataset)))


## 3. Initialization strategies

---
### Model 1: Shallow, Init = zeros

This section evaluates the performance when we use **zero** sampling for initializating a shallow network with 2 blocks.

| Network |Evaluation|  #Blocks | Activation | Initialization | BN? | Dropout? | 
|:--- |:---|:---:|:---:|:---:|:---:|:---:|
|1 | Shallow, Initialization| 2 | Sgmoid | Zeros | - | - | 

<font color="blue">

#### Exercise:
    
In this exercise, we shall build a model with the following network architecture:
    
<br><center><b><font color="black">Network 1</font></b></center>

|Block | Layer  |  Module/Function | Settings|
|:---:|:---:|:---:|:---:|
|1| 1  | Linear | #units = 50 |
|| -  | Sigmoid | - |
|2| 2  | Linear | #units = 10 |

    
Study the code below to learn how to initialize the parameters of a network. Update Table 1 above with the test result for model 1.     

Expected result: *Final training loss: approximately 2.0574*
</font>

In [ ]:
class Network1(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3*32*32, 50)
        self.fc2 = nn.Linear(50, 10)
        
        nn.init.zeros_(self.fc1.weight)
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)
                
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.sigmoid(self.fc1(x))
        x = self.fc2(x)
        return x
    
print(Network1())

Train model 1

In [ ]:
random_seed()
model1 = Network1()  
train(model1)

Evaluate model 1

In [ ]:
evaluate(model1)

Update Table 1 above with your result for model 1

---
### Model 2: Shallow, Init = normal 

This section evaluates the performance when we use **normal** sampling for initializating a shallow network.

|Model |Evaluation|  #Blocks | Activation | Initialization | BN? | Dropout? |
|:---|:---|:---:|:---:|:---:|:---:|:---:|
|2|Shallow, Initialization| 2 | Sigmoid | Normal | - | - | 

<font color="blue">

#### Exercise:
    
Model 2 has the same archiecture as model 1 above. However, the weights are initialized differently:
1. Initialize the **weights** with **normal** sampling (`nn.init.normal_`). 
2. Initialize the **bias** to **zero** (`nn.init.zeros_`). 

Then, update Table 1 above with the test result for model 2.

Expected result: *Final training loss: approximately 1.8800*

</font>

In [ ]:
class Network2(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(3*32*32, 50)
        self.fc2 = nn.Linear(50, 10)
        
        # Initialize the weights and bias
        ### START CODE HERE ### (≈ 4 lines of code)
        # ... your code here ...
        ### END OF CODE HERE ###
                
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.sigmoid(self.fc1(x))
        x = self.fc2(x)
        return x
    
print(Network2())

Train model 2

In [ ]:
random_seed()
model2 = Network2()   
train(model2)

Evaluate model 2

In [ ]:
evaluate(model2)

Update Table 1 above with your result for model 2

---
### Model 3: Shallow, Init = xavier_normal

This section evaluates the performance when we use **xavier normal** sampling for initializating a shallow network.

|Model |Evaluation|  #Blocks | Activation | Initialization | BN? | Dropout? | 
|:---:|:---:|:---|:---:|:---:|:---:|:---:|
|3|Shallow, Initialization| 2 | Sigmoid | Xavier_normal | - | - | 

<font color="blue">

#### Exercise:
    
Model3 has the same archiecture as model 1 above. However, the weights are initialized differently:
1. Initialize the **weights** with **xavier normal** sampling (`nn.init.xavier_normal_`). 
2. Initialize the **bias** to **zero** (`nn.init.zeros_`). 

Update Table 1 above with your test result for model 3.

Expected result: *Final training loss: approximately 1.4 to 1.5*

</font>

In [ ]:
class Network3(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3*32*32, 50)
        self.fc2 = nn.Linear(50, 10)
        
        # Initialize the weights and bias
        ### START CODE HERE ### (≈ 4 lines of code)
        # ... your code here ...
        ### END OF CODE HERE ###
                                               
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.sigmoid(self.fc1(x))
        x = self.fc2(x)
        return x
    
print(Network3())

Train model 3

In [ ]:
random_seed()
model3 = Network3()   
train(model3)

Evaluate model 3

In [ ]:
evaluate(model3)

---
### Model 4: Shallow, Init = kaiming_normal

This section evaluates the performance when we use **kaiming sampling** for initializating a shallow network.


|Model|Evaluation|  #Blocks | Activation | Initialization | BN? | Dropout? | 
|:---:|:---|:---:|:---:|:---:|:---:|:---:|
|4|Shallow, Initialization| 2 | Sigmoid | Kaiming Normal | - | - |

<font color="blue">

#### Exercise:
        
Model 4 has the same archiecture as model 1 above. However, the weights are initialized differently:
1. Initialize the **weights** with **kaiming normal** sampling (`nn.init.kaiming_normal_`). 
2. Initialize the **bias** to **zero** (`nn.init.zeros_`).

Update Table 1 above with the test result for model 4.

Expected result: *Final training loss: approximately 1.4 to 1.5*

</font>

In [ ]:
class Network4(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3*32*32, 50)
        self.fc2 = nn.Linear(50, 10)
        
        # Initialize the weights and bias
        ### START CODE HERE ### (≈ 4 lines of code)
        # ... your code here ...
        ### END OF CODE HERE ###
                        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.sigmoid(self.fc1(x))
        x = self.fc2(x)
        return x
    
print(Network4())

Train model 4

In [ ]:
random_seed()
model4 = Network4()   
train(model4)

Evaluate model 4

In [ ]:
evaluate(model4)

---
### Model 5: Deep, Init = kaiming_normal

The following code evaluates model one that initializes the parameters to xavier normal for a **deep** neural network. Model 5 have 6 blocks instead of 2 for previous models.

|Model|Evaluation|  #Blocks | Activation | Initialization | BN? | Dropout? |
|:---:|:---|:---:|:---:|:---:|:---:|:---:|
|5|Deep, Initialization| 6 | Sigmoid | kaiming_normal | - | - | 

<font color="blue">

#### Exercise:
    
The following code builds a **deep** neural network with **6 blocks**. 

<br><center><b><font color="black">Model 5</font></b></center>

|Block|Layer | Module/Function  | Settings |
|:---:|:---:|:---:|:---:|
|1|1 | Linear | #units = 50 |
||- | Sigmoid |-  |
|||||
|2|2 | Linear | #units = 50 |
||- | Sigmoid |-  |
|||||
|3|3 | Linear | #units = 50 |
||- | Sigmoid |-  |
|||||
|4|4 | Linear | #units = 50 |
||- | Sigmoid |-  |
|||||
|5|5 | Linear | #units = 50 |
||- | Sigmoid |-  |
|6|6 | Linear | #units = 50 |
    
    
A `for` loop is used to initialize the weights (with kaiming_normal) and biases (with zeros) for all layers. It iterates through all *modules* (`self.modules`) and initializes modules that are linear layers. Note that a module can be either a layer, a block (of layers) or the whole network itself. 

Study the code below to learn how to use a `for` loop to initialize all layers. Then update Table 1 with the result for model 5.

Expected result: *Final training loss: approximately 2.0295*

</font>

In [ ]:
class Network5(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.fc1 = nn.Linear(3*32*32, 50)
        self.fc2 = nn.Linear(50, 50)
        self.fc3 = nn.Linear(50, 50)
        self.fc4 = nn.Linear(50, 50)
        self.fc5 = nn.Linear(50, 50)
        self.fc6 = nn.Linear(50, 10)
        
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)
                        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.sigmoid(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        x = torch.sigmoid(self.fc4(x))
        x = torch.sigmoid(self.fc5(x))
        x = self.fc6(x)
        return x
    
print(Network5())

Train model 5

In [ ]:
random_seed()
model5 = Network5()   
train(model5)

Evaluate model 5

In [ ]:
evaluate(model5)

When the network grows **deeper** from 2 to 6 blocks, the network **could not train** properly even with kaiming normal initialization. In order to train the network, good initialization is no longer sufficient. 

To train a deep network, in the following, we are going to learn several strategies that allow us to train deep networks. This include 
1. "standardizing" the hidden layers by adding batch normalization and 
2. replacing the activation with "ReLU" which is less susceptible to saturating during backpropagation. 

---
## 4. Batch Normalization

To allow the network to train deeper, using xavier initialization alone is not sufficient. Two common ways to train deeper network is by adding **Batch Normalization** layers and use **ReLU** activation function rather than *Sigmoid*

In a deep neural network, the activation tends to get washed away at the deeper layers where the standard deviation of activation values becomes increasingly smaller. As a result, the neurons becomes increasingly similar at later layers. In addition, neurons becomes saturated when the weight is too small or big for normal initialization, thus causing **vanishing gradient effect** where the gradient becomes increasingly close to 0 as the gradient backpropagates from the loss layer to the input layers. 

**Batch normalization** avoids this by standardizing the output of each layer. When inserting a batch normalization (`bn`) layer into the neural network, it is common to insert it between the *linear* layer and the *activation* layer.

```
[LINEAR -> BATCH NORMALIZATION -> ACTIVATION]
```

### Model 6: Deep, BatchNorm, Sigmoid Activation

In the following, we shall evaluate adding batch normalization into a deep architecture with sigmoid activation for all hidden layers. Since a `linear` layer outputs a 1-dimensional data, we shall use `nn.BatchNorm1d` module. Refer to [PyTorch's documentation](https://pytorch.org/docs/stable/nn.html#batchnorm1d) to learn how to use the module.

|Model |Evaluation|  #Blocks | Activation | Initialization | BN? | Dropout? | 
|:---:|:---|:---:|:---:|:---:|:---:|:---:|
| 6 |Deep, Initialization| 6 | Sigmoid | kaiming_normal | Yes | None |

<font color="blue">

#### Exercise:
    
Build the network as shown below. Then, update Table 1 with the result for model 6.

<br><center><b><font color="black">Model 6</font></b></center>

| Block|Layer | Module/Function  | Settings |
|:---:|:---:|:---:|:---:|
|1|1 | Linear | #units = 50 |
||2 | Batch Normalization  | num_features=50  |
||- | Sigmoid |-  |
|||||
|2|3 | Linear | #units = 50 |
||4 | Batch Normalization  |  num_features=50  |
||- | Sigmoid |-  |
|||||
|3|5 | Linear | #units = 50 |
||6 | Batch Normalization  |  num_features=50  |
||- | Sigmoid |-  |
|||||
|4|7 | Linear | #units = 50 |
||8 | Batch Normalization  |  num_features=50  |
||- | Sigmoid |-  |
|||||
|5|9 | Linear | #units = 50 |
||10 | Batch Normalization  |  num_features=50  |
||- | Sigmoid |-  |
|||||
|6|11 | Linear | #units = 50 |
    
The `num_features` setting for the batch normalization must be set as #units in the previous linear layer

Expected result: 

```
Network6(
  (fc1): Linear(in_features=3072, out_features=50, bias=True)
  (bn1): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=50, out_features=50, bias=True)
  (bn2): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc3): Linear(in_features=50, out_features=50, bias=True)
  (bn3): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc4): Linear(in_features=50, out_features=50, bias=True)
  (bn4): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc5): Linear(in_features=50, out_features=50, bias=True)
  (bn5): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc6): Linear(in_features=50, out_features=10, bias=True)
)
```
*Final training loss: approximately 1.4 to 1.5*

    
</font>

In [ ]:
class Network6(nn.Module):
    def __init__(self):
        super().__init__()
        
        ### START CODE HERE ### 
        
        # layer 1, 2: define fc1 and bn1 (2 lines)
        # ... your code here ...

        # layer 3, 4: define fc2 and bn2 (2 lines)
        # ... your code here ...

        # layer 5, 6: define fc3 and bn3 (2 lines)
        # ... your code here ...
        
        # layer 7, 8: define fc4 and bn4 (2 lines)
        # ... your code here ...
        
        # layer 9, 10: define fc5 and bn5 (2 lines)
        # ... your code here ...

        # layer 11: define fc6  (1 line)
        # ... your code here ...
        
        ### END CODE HERE ###
        
        # Initialize the weights and bias. 
        # The weights should be initialized using kaiming normal. 
        # The bias should be initialized to zeros.
        ### START CODE HERE ### (Add 4 new lines of code) 
        # ... your code here ...
        ### END CODE HERE ###
                
                
    def forward(self, x):
        
        x = x.view(x.size(0), -1)
        
        ### START CODE HERE ###  
        
        # Layer 1, 2: fc -> bn -> sigmoid (1-3 lines) 
        # ... your code here ...
        
        # Layer 3, 4: fc -> bn -> sigmoid (1-3 lines) 
        # ... your code here ...

        # Layer 5, 6: fc -> bn -> sigmoid (1-3 lines) 
        # ... your code here ...

        # Layer 7, 8: fc -> bn -> sigmoid (1-3 lines) 
        # ... your code here ...
        
        # Layer 9, 10: fc -> bn -> sigmoid (1-3 lines) 
        # ... your code here ...

        # Layer 11: fc (1 line) 
        # ... your code here ...

        ### END CODE HERE ###

        return x

print(Network6())

Train model 6

In [ ]:
random_seed()
model6 = Network6()   
train(model6)

Evaluate model 6

In [ ]:
evaluate(model6)

With the Batch Normalization (BN), we manage to get good performance again with the deep neural network. BN layers standardizes the output of the linear layer and this in turn allows the model to be trained properly.

---
## 5. ReLU Activation

### Model 7: Deep, ReLU Activation

In modern times, the **sigmoid** activation has been dropped in favour of **ReLU**. In this section, you shall see the superiority of ReLU over sigmoid where ReLU is able to train even in the absence of batch normalization. When sigmoid neuron was used previously, the network failed to train.  All modern neural network has dropped sigmoid activation in favour of ReLU for deep neural networks. The sigmoid activation is used only in the *output* layers. 

| Model |Evaluation|  #Blocks | Activation | Initialization | BN? | Dropout? | 
|:---:|:---|:---:|:---:|:---:|:---:|:---:|
|7|Deep, Initialization| 6 | ReLU | kaiming_normal | None | None | 

**Defining a list of modules using `nn.ModuleList`**
    
In the following, we shall use an alternative way to build our deep network. Since all layers are similar in structure, we shall learn how to create the layers using `nn.ModuleList`.
    
<font color="blue">

#### Exercise:
    
Go through the code below and understand how to build a network using nn.ModuleList. Network 7 uses **relu** (`torch.relu`) is used the activation function. Then, update Table 1 with the test result for model 7.
    
<br><center><b><font color="black">Model 7</font></b></center>
    
|Layer | Module/Function  | Settings |
|:---:|:---:|:---:|
|1 | Linear | #units = 50 |
| | ReLU |-  |
||||
|2 | Linear | #units = 50 |
|- | ReLU |-  |
||||
|3 | Linear | #units = 50 |
|- | ReLU |-  |
||||
|4 | Linear | #units = 50 |
|- | ReLU |-  |
||||
|5 | Linear | #units = 50 |
|- | ReLU |-  |
||||
|6 | Linear | #units = 50 |
    

Expected result: 

```
Network7(
  (fc1): Linear(in_features=3072, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=50, bias=True)
  (fc3): Linear(in_features=50, out_features=50, bias=True)
  (fc4): Linear(in_features=50, out_features=50, bias=True)
  (fc5): Linear(in_features=50, out_features=50, bias=True)
  (fc6): Linear(in_features=50, out_features=10, bias=True)
)
```

*Final training loss: approximately 1.1 to 1.3*

In [ ]:
class Network7(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Build the layers nn.Module List. layer_sizes define the #units in each layer
        layer_sizes = [32*32*3, 50, 50, 50, 50, 50, 10]
        self.fc = nn.ModuleList([nn.Linear(layer_sizes[i-1], layer_sizes[i]) for i in range(1, len(layer_sizes))])
        
        for fc in self.fc:
            nn.init.kaiming_normal_(fc.weight)
            nn.init.zeros_(fc.bias)     
                        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        
        # propagate through all layers using a for loop
        for i in range(0, len(self.fc)-1):
            x = torch.relu(self.fc[i](x))
        x = self.fc[-1](x)
        
        return x
    
print(Network7())

* **Line 6**: defines the number of units in each layer. The first value (32*32*3) is the input dimension.
* **Line 7**: PyTorch only register modules (`nn.module` objects) that are placed directly in the `__init__`. It will not be able to recognize any modules that appear in a list. To register modules using a list, use `nn.ModuleList`. `nn.ModuleList` can be indexed like a regular Python list, but the modules it contains are also properly registered and will be visible to the network. The list starts from i=1 because the first value in `layer_sizes` (32*32*32) is the input dimension, not a network layer.
* **Line 17-19**: Use a for loop to perform forward propagation.


Train model 7

In [ ]:
random_seed()
model7 = Network7()   
train(model7)

Evaluate model 7

In [ ]:
evaluate(model7)

The test result should show that switching the activation function from `sigmoid` to `relu` results in significantly better performance even without Batch Normalization (BN).

---
### Model 8: Deep, BatchNorm, ReLU Activation

Now, let's combine batch normalization and reLU and see if it will give us a superior performance.

|Model|Evaluation|  #Blocks | Activation | Initialization | BN? | Dropout? |
|:---:|:---|:---:|:---:|:---:|:---:|:---:|
|8|Deep, Initialization| 6 | ReLU | kaiming_normal | Yes | -|

<font color="blue">

#### Exercise:

Build the network as shown below. You should use both **batch normalization (BN)** and **relu activation** for your model. Insert BN layers after all fully connected layers (`fc1` to `fc5`) and before the activations. The last layer (`fc6`) does not have batch normalization nor activation. Then update Table 1 with the test result for model 8.

<br><center><b><font color="black">Model 8</font></b></center>
    
|Block |Layer | Module/Function  | Settings |
|:---:|:---:|:---:|:---:|
|1|1 | Linear | #units = 50 |
||2 | Batch Normalization  | num_features=50  |
||- | ReLU |-  |
|||||
|2|3 | Linear | #units = 50 |
||4 | Batch Normalization  |  num_features=50  |
||- | ReLU |-  |
|||||
|3|5 | Linear | #units = 50 |
||6 | Batch Normalization  |  num_features=50  |
||- | ReLU |-  |
|||||
|4|7 | Linear | #units = 50 |
||8 | Batch Normalization  |  num_features=50  |
||- | ReLU |-  |
|||||
|5|9 | Linear | #units = 50 |
||10 | Batch Normalization  |  num_features=50  |
||- | ReLU |-  |
|||||
|6|11 | Linear | #units = 50 |

Expected output:
```
Network8(
  (fc1): Linear(in_features=3072, out_features=50, bias=True)
  (bn1): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=50, out_features=50, bias=True)
  (bn2): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc3): Linear(in_features=50, out_features=50, bias=True)
  (bn3): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc4): Linear(in_features=50, out_features=50, bias=True)
  (bn4): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc5): Linear(in_features=50, out_features=50, bias=True)
  (bn5): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc6): Linear(in_features=50, out_features=10, bias=True)
)
```

*Final training loss: approximately 1.1 to 1.3*

</font>

In [ ]:
class Network8(nn.Module):
    def __init__(self):
        super().__init__()
        
        ### START CODE HERE ### (6 lines for linear, 5 lines for batch norm)         
        # ... your code here ...
        ### END CODE HERE ###
        
        
        # Initialize all layers
        ### START CODE HERE ### (4 lines) 
        # ... your code here ...
        ### END CODE HERE ###

                        
    def forward(self, x):
        ### START CODE HERE ### 
        # (7 to 18 lines - 1 line to flatten input, 6 lines for linear, 5 lines for bn, 5 lines for relu)  
        ### END CODE HERE ###
        
        return x
    
print(Network8())

Train model 8

In [ ]:
random_seed()
model8 = Network8()   
train(model8)

Evaluate model 8

In [ ]:
evaluate(model8)

---
## 6. Dropout

### Model 9: Deep, Dropout, ReLU Activation

Now, let's evaluate the performance dropout on the dataset. Dropout has been shown to provide regularization effect and reduces overfitting. We follow the structure of the Independent Component (IC) block proposed by Guangyong Chen (2019) and place **batch normalization** and **dropout** next to each other. Hence, each block contains the layers in the following sequence:

```
[LINEAR -> ACTIVATION -> BATCH NORMALIZATION -> DROPOUT]
```

|Model|Evaluation|  #Blocks | Activation | Initialization | BN? | Dropout? |
|:---:|:---|:---:|:---:|:---:|:---:|:---:|
|9|Deep, Initialization| 6 | ReLU | kaiming_normal | Yes | Yes |

We shall add dropout layers for all layers except for the output layer. 

<font color="blue">

#### Exercise:

Build the network shown below. You should add dropout (`nn.Dropout`) to all the first 5 blocks of the network (`fc1` to `fc5`) as well as the input layer (after you flatten the input) with a dropout rate of 0.1. The dropout layer should be inserted after activation. Then, update Table 1 with the result for model 9.

<br><center><b><font color="black">Model 9</font></b></center>

|Block 1|Layer | Module/Function  | Settings |
|:---:|:---:|:---:|:---:|
||- | Dropout  | p=0.1  |
|1|1 | Linear | #units = 50 |
||- | ReLU |-  |
||2 | Batch Normalization  | num_features=50  |
||- | Dropout  | p=0.1  |
|||||
|2|3 | Linear | #units = 50 |
||- | ReLU |-  |
||4 | Batch Normalization  | num_features=50  |
||- | Dropout  | p=0.1  |
|||||
|3|5 | Linear | #units = 50 |
||- | ReLU |-  |
||6 | Batch Normalization  | num_features=50  |
||- | Dropout  | p=0.1  |
|||||
|4|7 | Linear | #units = 50 |
||- | ReLU |-  |
||8 | Batch Normalization  | num_features=50  |
||- | Dropout  | p=0.1  |
|||||
|5|9 | Linear | #units = 50 |
||- | ReLU |-  |
||10 | Batch Normalization  | num_features=50  |
||- | Dropout  | p=0.1  |
|||||
|6|11 | Linear | #units = 10 |

    
Expected output: 
```
Network9(
  (dropout0): Dropout(p=0.1, inplace=False)
  (fc1): Linear(in_features=3072, out_features=50, bias=True)
  (bn1): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.1, inplace=False)
  (fc2): Linear(in_features=50, out_features=50, bias=True)
  (bn2): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.1, inplace=False)
  (fc3): Linear(in_features=50, out_features=50, bias=True)
  (bn3): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout3): Dropout(p=0.1, inplace=False)
  (fc4): Linear(in_features=50, out_features=50, bias=True)
  (bn4): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout4): Dropout(p=0.1, inplace=False)
  (fc5): Linear(in_features=50, out_features=50, bias=True)
  (bn5): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout5): Dropout(p=0.1, inplace=False)
  (fc6): Linear(in_features=50, out_features=10, bias=True)
)
``` 
*Final training loss: approximately 1.6-1.7*
</font>

In [ ]:
class Network9(nn.Module):
    def __init__(self):
        super().__init__()

        ### START CODE HERE ###

        # input layer: define dropout0 (1 line)
        # ... your code here ...
        
        # Block 1: define fc1, bn1 and dropout1 (3 lines)
        # ... your code here ...

        # Block 2: define fc2, bn2, and droput2 (3 lines)
        # ... your code here ...

        # Block 3: define fc3, bn3 and dropout3 (3 lines)
        # ... your code here ...

        # Block 4: define fc4, bn4 and dropout4 (3 lines)
        # ... your code here ...

        # Block 5: define fc5, bn5 and dropout5 (3 lines)
        # ... your code here ...

        # Output layer: define fc6 (1 line)
        # ... your code here ...
        ### END CODE HERE ####
        
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):

        ### START CODE HERE ### (Add 6 new lines)  
        # input layer: flatten -> dropout 0 (2 lines)
        # ... your code here ...
        
        # Block 1: fc1 -> relu -> bn1 -> dropout1 (1-4 lines)
        # ... your code here ...
        
        # Block 2: fc2 -> relu -> bn2 -> dropout2 (1-4 lines)
        # ... your code here ...
        
        # Block 3: fc3 -> relu -> bn3 -> dropout3 (1-4 lines)
        # ... your code here ...
        
        # Block 4: fc4 -> relu -> bn4 -> dropout4 (1-4 lines)
        # ... your code here ...
        
        # Block 5: fc5 -> relu -> bn5 -> dropout5 (1-4 lines)
        # ... your code here ...
        
        # Output layer: fc6 (1 line)
        # ... your code here ...

        ### END CODE HERE ###
        
        return x
    
print(Network9())

Train model 9

In [ ]:
random_seed()
model9 = Network9()   
train(model9)

Evaluate model 9

In [ ]:
evaluate(model9)